In [1]:
import findspark
import subprocess
import time

import numpy as np
import xgboost as xgb

from tqdm import tqdm

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
findspark.init()

In [3]:
spark = SparkSession.builder \
    .appName("US Accidents Analysis - Optimized for MSI GF63") \
    .master("local[10]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "6g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .config("spark.default.parallelism", "12") \
    .config("spark.network.timeout", "600s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.sql.broadcastTimeout", "300s") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .config("spark.python.worker.reuse", "true") \
    .getOrCreate()

spark

In [4]:
DATA_PATH = "Data/Feature_Engineered_Data"

In [5]:
train_df = spark.read.parquet(f"{DATA_PATH}/train")
test_df  = spark.read.parquet(f"{DATA_PATH}/test")

In [6]:
print("Train:", train_df.count())
print("Test:", test_df.count())

Train: 8458433
Test: 2115303


In [7]:
train_df = train_df.withColumn("label", col("Severity") - 1)
test_df  = test_df.withColumn("label", col("Severity") - 1)

In [8]:
train_df = train_df.select("features", "label").cache()
test_df  = test_df.select("features", "label").cache()

In [9]:
train_df.count()
test_df.count()

2115303

In [10]:
def spark_to_numpy_stream(df):
    """
    Convert Spark DF to NumPy WITHOUT collecting all data at once.
    Uses streaming via toLocalIterator()
    """

    X_list = []
    y_list = []

    for row in tqdm(df.toLocalIterator(), desc="Streaming data"):
        X_list.append(row["features"].toArray())
        y_list.append(row["label"])

    X = np.array(X_list)
    y = np.array(y_list)

    return X, y

In [11]:
print("Streaming training data...")
X_train, y_train = spark_to_numpy_stream(train_df)

Streaming training data...


Streaming data: 8458433it [05:54, 23827.64it/s]


In [12]:
print("Streaming test data...")
X_test, y_test = spark_to_numpy_stream(test_df)

Streaming test data...


Streaming data: 2115303it [01:25, 24659.29it/s]


In [13]:
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

Train shape: (8458433, 39)
Test shape : (2115303, 39)


In [14]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest  = xgb.DMatrix(X_test, label=y_test)

In [15]:
params = {
    "objective": "multi:softprob",
    "num_class": 4,

    # GPU
    "tree_method": "hist",
    "device": "cuda",
    "predictor": "gpu_predictor",

    # Model
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "eval_metric": "mlogloss"
}

In [16]:
num_rounds = 60

In [17]:
print("Training started on GPU...")

model = None

for i in tqdm(range(num_rounds), desc="XGBoost Training"):
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=i + 1,
        evals=[(dtest, "test")],
        xgb_model=model,
        verbose_eval=False
    )

print("Training completed.")

Training started on GPU...


XGBoost Training:   0%|          | 0/60 [00:00<?, ?it/s]C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:200: UserWarning: [19:05:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
XGBoost Training:   2%|▏         | 1/60 [00:03<03:51,  3.92s/it]C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:200: UserWarning: [19:05:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
XGBoost Training:   3%|▎         | 2/60 [00:06<02:44,  2.84s/it]C:\Users\LENOVO\AppData\Roaming\Python\Python311\site-packages\xgboost\training.py:200: UserWarning: [19:05:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "predictor" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
XGBoost T

Training completed.


In [18]:
y_pred_prob = model.predict(dtest)
y_pred = np.argmax(y_pred_prob, axis=1)

In [19]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average="weighted"))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8716250106958672
F1 Score: 0.8546180144583703

Classification Report:

              precision    recall  f1-score   support

           0       0.73      0.17      0.28     18526
           1       0.89      0.97      0.93   1739051
           2       0.72      0.50      0.59    290706
           3       0.65      0.17      0.27     67020

    accuracy                           0.87   2115303
   macro avg       0.75      0.45      0.52   2115303
weighted avg       0.86      0.87      0.85   2115303



In [24]:
model.save_model("Models/xgboost_model.json")
print("Model saved successfully.")

Model saved successfully.


In [25]:
print(subprocess.check_output("nvidia-smi", shell=True).decode())

Wed May  6 16:44:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.36                 Driver Version: 596.36         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:02:00.0 Off |                  N/A |
| N/A   38C    P3             13W /   30W |    2441MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----